# Pelabelan data

In [2]:
import pandas as pd

df = pd.read_csv("data_ulasan_mentah.csv")

def label(score):
  if score <= 2:
    return "negatif"
  elif score == 3:
    return "netral"
  else:
    return "positif"

df["label"] = df["score"].apply(label)

print("Jumlah data per kelas:")
print(df["label"].value_counts())

Jumlah data per kelas:
label
positif    7082
negatif    3172
netral      746
Name: count, dtype: int64


# Preprocessing Data

In [3]:
import re
import string

df['content'] = df['content'].fillna('').astype(str)

def clean_text(text):

  if not isinstance(text, str):
    return ""

  text = text.lower()
  text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
  text = re.sub(r"\@\w+|\#", "", text)
  text = re.sub(r"[^a-zA-Z\s]", "", text)
  text = re.sub(r"\s+", " ", text).strip()
  return text

df["text_clean"] = df["content"].apply(clean_text)
df = df[df["text_clean"] != ""]
df[['content', 'text_clean']].head()

,content,text_clean
0,game nya seru banget permainan nya juga banyak...,game nya seru banget permainan nya juga banyak...
1,tolong kasih fitur chat,tolong kasih fitur chat
2,Sering terjadi server error jadi tidak terlalu...,sering terjadi server error jadi tidak terlalu...
3,bagus banget aku ngak bisa berkata² pokok debe...,bagus banget aku ngak bisa berkata pokok debes...
4,KALO MAU BINTANG LIMA BENERIN DULU NOH GAME LUU🙄,kalo mau bintang lima benerin dulu noh game luu


# Normalisasi

In [4]:
slang_dict = {
    "gx": "tidak",
    "gk": "tidak",
    "gak": "tidak",
    "ngak": "tidak",
    "nggak": "tidak",
    "bgt": "banget",
    "bgs": "bagus",
    "bgus": "bagus",
    "ny": "nya",
    "game nya": "gamenya",
    "gamnya": "gamenya",
    "gem": "game",
    "klo": "kalau",
    "udh": "sudah",
    "sdh": "sudah",
    "pake": "pakai",
    "aj": "saja",
    "aja": "saja",
    "gw": "saya",
    "gua": "saya",
    "donk": "dong",
    "yaudah": "ya sudah",
    "knp" : "kenapa",
    "bnyak": "banyak",

}

def normalisasi_kata(text):
    words = text.split()
    normalized_words = [slang_dict.get(w, w) for w in words]
    return " ".join(normalized_words)

# Terapkan ke data yang sudah bersih tadi
df['text_normalized'] = df['text_clean'].apply(normalisasi_kata)

print("Hasil Normalisasi:")
df[['text_clean', 'text_normalized']].head()

Hasil Normalisasi:


,text_clean,text_normalized
0,game nya seru banget permainan nya juga banyak...,game nya seru banget permainan nya juga banyak...
1,tolong kasih fitur chat,tolong kasih fitur chat
2,sering terjadi server error jadi tidak terlalu...,sering terjadi server error jadi tidak terlalu...
3,bagus banget aku ngak bisa berkata pokok debes...,bagus banget aku tidak bisa berkata pokok debe...
4,kalo mau bintang lima benerin dulu noh game luu,kalo mau bintang lima benerin dulu noh game luu


# Label Encoding

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Positif -> 2, Netral -> 1, Negatif -> 0 (tergantung urutan abjad)
df['label_encoded'] = le.fit_transform(df['label'])

print("Mapping Label:")
for index, class_label in enumerate(le.classes_):
    print(f"{class_label} -> {index}")

Mapping Label:
negatif -> 0
netral -> 1
positif -> 2


# Data Splitting

In [6]:
from sklearn.model_selection import train_test_split


X = df['text_normalized']
y = df['label_encoded']

#80% training 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total Data Training: {len(X_train)}")
print(f"Total Data Testing: {len(X_test)}")

Total Data Training: 8714
Total Data Testing: 2179


# Oversampling

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

# Latih dan transformasi data TRAINING
X_train_tfidf = tfidf.fit_transform(X_train)

# Transformasi data TESTING 
X_test_tfidf = tfidf.transform(X_test)

In [9]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Inisialisasi SMOTE
smote = SMOTE(random_state=42)

# Terapkan SMOTE di data training yang sudah jadi angka
X_train_res, y_train_res = smote.fit_resample(X_train_tfidf, y_train)

print(f"Sebelum SMOTE: {Counter(y_train)}")
print(f"Sesudah SMOTE: {Counter(y_train_res)}")

Sebelum SMOTE: Counter({2: 5567, 0: 2535, 1: 612})
Sesudah SMOTE: Counter({0: 5567, 2: 5567, 1: 5567})


# Model Machine Learning
- Logistic Regression
- Naive Bayes
- Random Forest

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

model_dict = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in model_dict.items():
    # Training menggunakan data hasil SMOTE (X_train_res, y_train_res)
    model.fit(X_train_res, y_train_res)
    
    # Prediksi menggunakan data testing ASLI yang sudah di-TFIDF
    y_pred = model.predict(X_test_tfidf)
    
    # 3. Output Hasil (Untuk bukti Inforensi Kriteria 6)
    print(f"================ {name} ================")
    print(f"Akurasi Testing: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("\n")

================ Logistic Regression ================
Akurasi Testing: 0.7155
              precision    recall  f1-score   support

     negatif       0.69      0.68      0.68       621
      netral       0.12      0.33      0.18       129
     positif       0.90      0.77      0.83      1429

    accuracy                           0.72      2179
   macro avg       0.57      0.59      0.56      2179
weighted avg       0.79      0.72      0.75      2179



================ Naive Bayes ================
Akurasi Testing: 0.6732
              precision    recall  f1-score   support

     negatif       0.65      0.59      0.62       621
      netral       0.13      0.50      0.21       129
     positif       0.91      0.72      0.81      1429

    accuracy                           0.67      2179
   macro avg       0.56      0.60      0.55      2179
weighted avg       0.79      0.67      0.72      2179



================ Random Forest ================
Akurasi Testing: 0.7618
              

# Model Deep Learning